# Reading a plan

Read a Terraform plan, both as people see it and as the JSON that tools read, summarise what each change will create, update, replace or destroy, and do it for six real pull requests.

*From [CloudTech Academy](https://academy.cloudtechanalytics.com/learn/terraform-infrastructure-as-code/reading-a-plan). Run the cells in order (Runtime → Run all), edit them, and experiment. Answer the practice questions on the lesson page.*

## The problem

Six pull requests are waiting for review in Tallybook's infrastructure repository. Each has a title and a description written by its author, and a **plan** produced automatically by the pipeline: the list of everything that will happen if it's merged and applied.

The title is what the author **meant** to do. The plan is what **will** happen. Reviewing infrastructure means reading the plan, not the title.

## The concept

**Plan symbols**

| Symbol | Action in JSON | Meaning |
| :-- | :-- | :-- |
| `+` | `["create"]` | create a new resource |
| `~` | `["update"]` | change it in place |
| `-` | `["delete"]` | destroy it |
| `-/+` | `["delete", "create"]` | **replace**: destroy, then create a new one |
| `+/-` | `["create", "delete"]` | replace, creating the new one first |

The summary line counts a replacement as one add **and** one destroy: `Plan: 1 to add, 0 to change, 1 to destroy.`

**Plan JSON**

`terraform show -json plan.out` gives every change as data: the resource's address, the actions, and its attributes `before` and `after`. Review tools and policy checks (lesson 7) read this.

## Example

Load all six plans and summarise each the way Terraform's summary line does:

In [ ]:
import json
from urllib.request import urlopen

import pandas as pd

base = "https://academy.cloudtechanalytics.com/datasets/terraform/"
PRS = ["101-web-autoscaling", "102-rename-database", "103-reporting-access", "104-cost-tags", "105-rightsize-api", "106-multi-az-database"]

def load(name):
    with urlopen(base + name) as f:
        return json.load(f)

plans = {pr: load(f"plan-pr-{pr}.json") for pr in PRS}

def summarise(plan):
    add = change = destroy = 0
    for rc in plan["resource_changes"]:
        actions = rc["change"]["actions"]
        add += "create" in actions
        destroy += "delete" in actions
        change += actions == ["update"]
    return {"add": add, "change": change, "destroy": destroy}

pd.DataFrame({pr: summarise(p) for pr, p in plans.items()}).T

*Expected output:*

```
add  change  destroy
101-web-autoscaling      4       0        6
102-rename-database      1       0        1
103-reporting-access     1       0        0
104-cost-tags            0      19        0
105-rightsize-api        0       4        0
106-multi-az-database    0       1        0
```

Read across the table before reading any titles. PR 104 changes many resources but adds and destroys nothing: probably low risk. PR 101 destroys six resources. PR 102, "rename database", adds one and destroys one. Look at what exactly:

In [ ]:
for pr in ["101-web-autoscaling", "102-rename-database"]:
    print(f"PR {pr}:")
    for rc in plans[pr]["resource_changes"]:
        print("  ", "/".join(rc["change"]["actions"]).ljust(14), rc["address"])

*Expected output:*

```
PR 101-web-autoscaling:
   create         aws_launch_template.web
   create         aws_autoscaling_group.web
   create         aws_autoscaling_policy.web_cpu
   create         aws_autoscaling_schedule.month_end
   delete         aws_instance.web[0]
   delete         aws_instance.web[1]
   delete         aws_instance.web[2]
   delete         aws_instance.web[3]
   delete         aws_instance.web[4]
   delete         aws_instance.web[5]
PR 102-rename-database:
   delete         aws_db_instance.prod
   create         aws_db_instance.main
```

PR 101 creates an autoscaling group (the cloud course's recommendation) and deletes the six fixed web servers, which is intended, but all at once. PR 102 deletes `aws_db_instance.prod` and creates `aws_db_instance.main`. A rename in code is, to Terraform, a deletion of one resource and the creation of another. Applied, it would **destroy the production database**. Lesson 6 is about catching and fixing exactly this.

## Walkthrough

1. Run the cells. List PR 104's changes. Which attribute changes on each resource?
2. List PR 106's single change. Is anything changing besides what the title says?
3. For each PR, write one sentence on what it does, based only on its plan.
4. Rank the six PRs from riskiest to safest (the task below).

## Practice

**Practice:** How many resources does PR 104 (cost tags) **change** in place?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Task:** Rank the **six PRs from riskiest to safest**, one per numbered line starting with the PR number, each with the **reason from its plan** (what it creates, changes or destroys).

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## Check your understanding